In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import joblib
import os

# Create folder if not exists
if not os.path.exists('data'):
    os.makedirs('data')

def run_full_project():
    print("--- 🚀 Project Started: Customer Churn & Revenue System ---")

    # --- TASK 1: Data Loading ---
    file_path = 'data/customer_churn.csv'
    if not os.path.exists(file_path):
        print("❌ Error: data/customer_churn.csv nahi mili. Pehle CSV file banayein.")
        return
    
    df = pd.read_csv(file_path)
    print("\n✅ Task 1: Data Loaded Successfully")
    print(df.info())

    # --- TASK 2: Data Cleaning ---
    df.dropna(inplace=True)
    # Encoding (pd.get_dummies) as per image instruction
    df_encoded = pd.get_dummies(df, columns=['Contract', 'InternetService'], drop_first=True)
    # Convert Churn to Numeric
    df_encoded['Churn'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})
    print("✅ Task 2: Data Cleaning & Encoding Done")

    # --- TASK 3: Exploratory Data Analysis (EDA) ---
    print("\n📊 Task 3: Analyzing Churn Patterns...")
    # 1. Churn by Contract
    contract_churn = df.groupby("Contract")["Churn"].value_counts()
    print("Churn by Contract Type:\n", contract_churn)

    # 2. Correlation Plot
    plt.figure(figsize=(10, 6))
    sns.heatmap(df_encoded.corr(), annot=True, cmap='RdYlGn')
    plt.title('Feature Correlation Matrix')
    plt.show()

    # --- TASK 4 & 5: Churn Prediction & Evaluation ---
    X = df_encoded.drop(['Churn', 'CustomerID', 'TotalCharges'], axis=1)
    y = df_encoded['Churn']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    churn_model = RandomForestClassifier(n_estimators=100)
    churn_model.fit(X_train, y_train)
    
    y_pred = churn_model.predict(X_test)
    print("\n✅ Task 4 & 5: Model Training & Evaluation")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
    print(f"F1 Score: {f1_score(y_test, y_pred):.2f}")
    
    # Save Churn Model
    joblib.dump(churn_model, 'churn_model.pkl')

    # --- TASK 6: Revenue Forecasting ---
    X_rev = df_encoded[['Tenure', 'MonthlyCharges']]
    y_rev = df_encoded['TotalCharges']
    
    rev_model = LinearRegression()
    rev_model.fit(X_rev, y_rev)
    
    # Save Revenue Model
    joblib.dump(rev_model, 'revenue_model.pkl')
    print("✅ Task 6: Revenue Forecasting Model Trained")

    # --- OPTIONAL TASK: Risk Segmentation ---
    probs = churn_model.predict_proba(X_test)[:, 1]
    
    def get_risk(p):
        if p < 0.30: return "Low Risk"
        elif p < 0.60: return "Medium Risk"
        else: return "High Risk"

    print("\n🎯 Optional Task: Customer Risk Segmentation (Sample)")
    sample_results = pd.DataFrame({
        'Prob': probs,
        'Risk': [get_risk(p) for p in probs]
    }).head(5)
    print(sample_results)

    print("\n✨ All Tasks Completed! Models are saved as .pkl files.")

if __name__ == "_main_":
    run_full_project()